# 更高级的练习：三方对话（Multi-bot Conversation）

## 练习目标

试着做一个**三方对话**：让三个不同性格的机器人轮流发言（也可以引入 Gemini）。社区贡献文件夹里已有同学的实现可供对照。

最稳妥的提示写法是：每一轮只给模型 **1 条 system + 1 条 user**，并在 user 里粘贴「迄今为止的完整对话」。

示意（逻辑示意；真正可运行的英文 prompt 见下方代码格）：

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything
in the conversation and challenge everything in a snarky way.
You are talking with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now respond with what you would like to say next, as Alex.
"""
```

建议先自己动手实现，再对照解法。用 OpenAI Python 客户端走兼容接口访问 Gemini 往往最简单（参见本课前面的 Gemini 示例）。

## 附加练习

也可以把其中某个角色换成 **Ollama** 本地开源模型（本笔记本正是这样做的）。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages` = system + user | 每轮只发两条，历史塞进 user |
| 多模型 | `llama3.2:1b` / `phi` / `qwen2.5:3b` |
| 对话上下文 | 共享字符串 `conversation`，每轮追加并裁剪 |


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务相关性</h2>
            <span style="color:#181;">这种「对话历史作为上下文」的结构，是构建对话式 AI 助手的核心：助手必须在多轮中记住已说内容。后续实验会用同样模式做助手，你也可以迁移到自己的业务场景。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 三方辩论机器人：三个本地模型轮流发言 ==========

# 从 openai 导入 OpenAI：通过 OpenAI 兼容协议调用本地 Ollama
from openai import OpenAI

# 创建客户端：base_url 指向本机 Ollama /v1；api_key 仅占位
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# bots：三个角色，各自绑定不同 model 与英文 system prompt（prompt 原文必须保留）
bots = [
    {
        "name": "Alex",
        "model": "llama3.2:1b",
        "system": """
You are Alex, a chatbot who is very argumentative.
You disagree with everything and challenge everything in a snarky way.
You are in a conversation with Blake and Charlie.
"""
    },
    {
        "name": "Blake",
        "model": "phi",
        "system": """
You are Blake, a polite and courteous chatbot.
You try to agree with everything and find common ground.
You calm down arguments.
You are in a conversation with Alex and Charlie.
"""
    },
    {
        "name": "Charlie",
        "model": "qwen2.5:3b",
        "system": """
You are Charlie, an analytical thinker.
You evaluate both sides and provide balanced, logical insights.
You are in a conversation with Alex and Blake.
"""
    }
]

# 共享对话历史（纯文本）；先写入用户开场议题（英文保持原样）
conversation = "User: Should companies adopt AI aggressively?\n"


# call_bot：让某一个 bot 基于当前 conversation 生成下一句
def call_bot(bot, conversation):
    # 组装 user_prompt：把完整历史塞进提示（英文模板保持原样）
    user_prompt = f"""
You are {bot['name']}, in conversation with the others.

The conversation so far is as follows:
{conversation}

Now respond with what you would like to say next, as {bot['name']}.
"""

    # Chat Completions：system 定人设，user 带历史
    response = client.chat.completions.create(
        model=bot["model"],  
        messages=[
            {"role": "system", "content": bot["system"]},
            {"role": "user", "content": user_prompt}
        ]
    )

    # 返回助手文本
    return response.choices[0].message.content


# 外层：共 5 轮；每轮三个 bot 各说一次
for i in range(5):
    # 打印轮次分隔（文案保持原样）
    print(f"\n--- Round {i+1} ---\n")

    # 依次让 Alex → Blake → Charlie 发言
    for bot in bots:
        # 基于当前共享历史生成回复
        reply = call_bot(bot, conversation)

        # 打印「名字 (模型): 回复」
        print(f"{bot['name']} ({bot['model']}): {reply}\n")

        # 把该角色发言追加到共享历史
        conversation += f"{bot['name']}: {reply}\n"

        # 只保留最近约 30 行，控制上下文长度与延迟
        conversation = "\n".join(conversation.split("\n")[-30:])



--- Round 1 ---

Alex (llama3.2:1b): Finally, Blake gets around to asking something worthwhile. I'd love to see some real thought put into this question.

Companies adopting AI aggressively? Are you kidding me? It's a recipe for disaster. We're talking job losses on a massive scale, automation of entire industries, and the loss of human dignity as companies try to insulate themselves from the consequences of their own recklessness. Newsflash: AI is not a silver bullet, it's a speeding bullet going off a cliff, and everyone involved should be staring in horror at how far down this path they've headed.

What Blake and Charlie are probably thinking is that "companies would always figure out some way to use AI for their own benefit", but what if they're right? What if these companies really are incapable of innovation without outside interference? Well, then we should be having a more nuanced discussion about why this is the case. Maybe Blake and Charlie genuinely believe we should just l